# Attack Regularizer
Visualises the effect of *targeted* FTLE flossing on a specific vulnerable input point.

**Workflow:**
1. Train a standard (task-only) control model.
2. Find a successfully attacked point with the highest FTLE — the *victim*.
3. FTLE-flosss *only* the victim point and its 36 equidistant perturbations.
4. Compare trajectories, decision boundaries, FTLE heatmaps, and attack success before vs. after targeted flossing.

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import os
import importlib
from torch.utils.data import TensorDataset, DataLoader
from IPython.display import Image, HTML
import time

import plots.plots
from plots.plots import set_plot_defaults
set_plot_defaults()

%matplotlib inline
%config InlineBackend.figure_formats = ['svg']

device = torch.device('cpu')
seed = 67
torch.manual_seed(seed)

subfolder = 'attack_regularizer_victim'
os.makedirs(subfolder, exist_ok=True)
print('subfolder:', subfolder)

## Data

In [ ]:
import models.training
importlib.reload(models.training)
from models.training import create_dataloader

cross_entropy = False
output_dim = 2
data_noise = 0.05
num_points = 5000
batch_size = 30
plotlim = [-2, 2]

dataloader, dataloader_test = create_dataloader(
    'moons', batch_size=batch_size, noise=data_noise, cross_entropy=cross_entropy,
    num_points=num_points, plotlim=plotlim, random_state=seed,
    label='vector', filename=subfolder + '/trainingset'
)

# Collect full test set as tensors
X_list, y_list = [], []
for X_b, y_b in dataloader_test:
    X_list.append(X_b)
    y_list.append(y_b)
X_test = torch.cat(X_list)
y_test = torch.cat(y_list)
print(f'Test set: {X_test.shape}, {y_test.shape}')

## Control model — task-only training

In [ ]:
from models.neural_odes import NeuralODEvar

hidden_dim, data_dim = 5, 2
T, time_steps = 10, 10
num_params = 1
layers_hidden = 1
non_linearity = 'tanh'
architecture = 'inside'
epochs_control = 30

load_model = True

torch.manual_seed(seed)
anode_control = NeuralODEvar(
    device, data_dim, hidden_dim, output_dim=output_dim,
    non_linearity=non_linearity, architecture=architecture,
    T=T, time_steps=time_steps, num_params=num_params,
    layers_hidden=layers_hidden, cross_entropy=cross_entropy
)
print('Control model created.')

In [ ]:
importlib.reload(models.training)
from models.training import doublebackTrainer, save_model, save_history

optimizer_control = torch.optim.Adam(anode_control.parameters(), lr=1e-3)
trainer_control = doublebackTrainer(
    anode_control, optimizer_control, device,
    cross_entropy=cross_entropy, turnpike=False, verbose=True
)

if load_model:
    model_path = subfolder + '/control_model.pth'
    if os.path.exists(model_path):
        anode_control.load_state_dict(torch.load(model_path)['state_dict'])
        print('Loaded control model from', model_path)
    else:
        print('No saved control model found at', model_path)
else:
    trainer_control.train(dataloader, num_epochs=epochs_control)
    save_model(anode_control, subfolder + '/control_model.pth')
    save_history(trainer_control.histories, subfolder + '/control_history.json')
print('Done.')

In [ ]:
importlib.reload(plots.plots)
from plots.plots import classification_levelsets

classification_levelsets(
    anode_control,
    footnote=f'control ({epochs_control} epochs), seed={seed}'
)

## Find successfully attacked point with highest FTLE

In [ ]:
importlib.reload(plots.plots)
from plots.plots import find_attacks

adversarial_budget = 0.3
equi_amount = 36
margin = 0.071

X_base, y_base, grad_best, stats = find_attacks(
    anode_control, X_test, y_test,
    adversarial_budget=adversarial_budget,
    attack_type='equi', equi_amount=equi_amount,
    margin=margin, return_stats=True, verbose=True
)
print(f'High-confidence successful attacks: {len(X_base)}')

In [ ]:
from FTLE import LEs

time_interval = torch.tensor([0., 5.])

print('Computing FTLEs for attacked base points...')
ftle_values = []
for pt in X_base:
    les = LEs(pt, anode_control, time_interval=time_interval)
    ftle_values.append(les[0].item())
ftle_values = torch.tensor(ftle_values)
print(ftle_values)

best_idx = int(ftle_values.argmax())
# best_idx = 1 #int(np.random.randint(0, 20))
print(best_idx)
victim_point = X_base[best_idx].clone()   # (2,)
victim_label = y_base[best_idx].clone()   # (2,)
victim_grad  = grad_best[best_idx].clone()  # (2,)  — best equi attack direction

print(f'Victim point:      {victim_point.numpy()}')
print(f'Victim FTLE (max): {ftle_values[best_idx]:.4f}')
print(f'Attack direction:  {victim_grad.numpy()}')

## Build flossing dataset — victim + all 36 equi perturbations

In [ ]:
reg_perturbations = [True, False]  # True: floss victim + perturbations; False: floss victim only (repeated)

angles = torch.linspace(0, 2 * torch.pi, equi_amount + 1)[:-1]  # (36,)
directions = torch.stack([torch.sin(angles), torch.cos(angles)], dim=1)  # (36, 2)
perturbations = victim_point.unsqueeze(0) + adversarial_budget * directions  # (36, 2)

# Full set — always used for trajectory visualisation
flosss_points = torch.cat([victim_point.unsqueeze(0), perturbations], dim=0)  # (37, 2)
flosss_labels = victim_label.unsqueeze(0).expand(len(flosss_points), -1).clone()  # (37, 2)

n_total = len(flosss_points)  # 37 — same total reg calls in both modes

# Per-option flossing loaders
flosss_loaders = {}
for reg_pert in reg_perturbations:
    if reg_pert:
        pts  = flosss_points
        lbls = flosss_labels
        batch = n_total                                              # 1 step/epoch, 37 pts per FTLE call
    else:
        pts  = victim_point.unsqueeze(0).expand(n_total, -1).clone() # (37, 2) — victim repeated
        lbls = victim_label.unsqueeze(0).expand(n_total, -1).clone() # (37, 2)
        batch = 1                                                    # 37 steps/epoch, victim each time
    flosss_loaders[reg_pert] = DataLoader(TensorDataset(pts, lbls), batch_size=batch)
    print(f'reg_perturbations={reg_pert}: {len(pts)} flossing point(s), batch_size={batch}')

## Visualisation set — victim + all equi perturbations

In [ ]:
# Reuse the flossing set: victim + all 36 equi-directional perturbations.
# All points carry the victim's original label so the trajectory colour reflects
# the true class, not the model prediction — perturbations that cross the decision
# boundary will visibly land in the wrong region.
viz_points = flosss_points.clone()   # (37, 2)
viz_labels = flosss_labels.clone()   # (37, 2) — all victim's original label
print(f'Visualisation set: {len(viz_points)} points (victim + {equi_amount} equi perturbations)')

## Trajectory — before flossing

In [ ]:
importlib.reload(plots.plots)
from plots.plots import trajectory_gif_new, trajectory_gif_attacks

xrange = [-4,9]
yrange = [-3,6]

trajectory_gif_attacks(
    anode_control, viz_points, viz_labels,
    timesteps=time_steps,
    filename=subfolder + '/trajectory_before.gif',
    dpi=200, xrange=xrange, yrange=yrange,
    title='regular training',
    footnote=f'control ({epochs_control} epochs)',
    colorbar=True
)
Image(subfolder + '/trajectory_before.gif')

## Targeted FTLE flossing — three intervals

In [ ]:
importlib.reload(models.training)
from models.training import load_model, FTLE_flossing

le_threshold = 0.3
epochs_flosss = 10
le_reg = 1.



flosss_intervals = [
    torch.tensor([0., 2.]),           # [0, 2]
    torch.tensor([0., 4.]),           # [0, 4]
    torch.tensor([0., 6.]),           # [0, 6]
    torch.tensor([0., 8.]),           # [0, 8]
    torch.tensor([0., 9.]),           # [0, 9]
    torch.tensor([0., float(T)]),     # [0, 10]
]
interval_labels = ['[0, 2]', '[0, 4]', '[0, 6]', '[0, 8]', '[0, 9]', '[0, 10]']
interval_safe   = ['0_2',    '0_4',    '0_6',    '0_8',    '0_9',    '0_10']

reg_pert_labels = {True: 'with perturbations', False: 'victim only'}
reg_pert_safe   = {True: 'p',                  False: 'v'}

print(f'Flossing {epochs_flosss} epochs, threshold={le_threshold}')

anodes_flossed = {reg_pert: [] for reg_pert in reg_perturbations}
for reg_pert in reg_perturbations:
    loader = flosss_loaders[reg_pert]
    print(f'\n### reg_perturbations = {reg_pert} ({len(loader.dataset)} input(s)) ###')
    for interval, lbl in zip(flosss_intervals, interval_labels):
        print(f'\n=== interval {lbl} ===')
        anode = load_model(subfolder + '/control_model.pth')
        trainer = FTLE_flossing(anode, device=device)
        trainer.train(loader, num_epochs=epochs_flosss, mode='ftle',
                      threshold=le_threshold, interval=interval, le_reg = le_reg)
        anodes_flossed[reg_pert].append(anode)

print('\nDone.')

## Trajectories — after flossing

In [ ]:
importlib.reload(plots.plots)
from plots.plots import trajectory_gif_attacks

for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    for anode, lbl, safe in zip(anodes_flossed[reg_pert], interval_labels, interval_safe):
        trajectory_gif_attacks(
            anode, viz_points, viz_labels,
            timesteps=time_steps,
            filename=subfolder + f'/trajectory_after_{safe}_{safe_p}.gif',
            xrange=xrange, yrange=yrange, dpi=200,
            title=f'reg interval = {lbl}',
            footnote=f'threshold={le_threshold}, flossing epochs={epochs_flosss}, seed = {seed}'
        )
        print(f'Generated trajectory for {lbl} / {lbl_p}')

last_frame = time_steps - 1
before_html = (
    f'<figure style="margin:4px"><figcaption style="font-size:10px">before flossing</figcaption>'
    f'<img src="{subfolder}/trajectory_before{last_frame}.png?{time.time()}" width="280"></figure>'
)
after_html = ''
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    after_html += ''.join(
        f'<figure style="margin:4px"><figcaption style="font-size:10px">after flossing {lbl} ({lbl_p})</figcaption>'
        f'<img src="{subfolder}/trajectory_after_{safe}_{safe_p}{last_frame}.png?{time.time()}" width="280"></figure>'
        for lbl, safe in zip(interval_labels, interval_safe)
    )
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{before_html}{after_html}</div>'))

## Decision boundaries — before vs after

In [ ]:
importlib.reload(plots.plots)
from plots.plots import classification_levelsets

classification_levelsets(
    anode_control,
    fig_name=subfolder + '/levelsets_control',
    footnote=f'control ({epochs_control} epochs)'
)
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    for anode, lbl, safe in zip(anodes_flossed[reg_pert], interval_labels, interval_safe):
        classification_levelsets(
            anode,
            fig_name=subfolder + f'/levelsets_flossed_{safe}_{safe_p}',
            footnote=f'targeted flossing {lbl} ({lbl_p}, {epochs_flosss} epochs, threshold={le_threshold})'
        )

control_html = (
    f'<figure style="margin:4px"><figcaption style="font-size:10px">control</figcaption>'
    f'<img src="{subfolder}/levelsets_control.png?{time.time()}" width="280"></figure>'
)
imgs_html = ''
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    imgs_html += ''.join(
        f'<figure style="margin:4px"><figcaption style="font-size:10px">flossing {lbl} ({lbl_p})</figcaption>'
        f'<img src="{subfolder}/levelsets_flossed_{safe}_{safe_p}.png?{time.time()}" width="280"></figure>'
        for lbl, safe in zip(interval_labels, interval_safe)
    )
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{control_html}{imgs_html}</div>'))

## FTLE heatmaps — before vs after

In [ ]:
from FTLE import LE_grid

le_density = 50
print('Computing FTLE grids...')
output_max_control, _ = LE_grid(anode_control, x_amount=le_density, time_interval=time_interval)
le_grids = {reg_pert: [] for reg_pert in reg_perturbations}
for reg_pert in reg_perturbations:
    print(f'  reg_pert={reg_pert}')
    for anode, lbl in zip(anodes_flossed[reg_pert], interval_labels):
        print(f'    {lbl}')
        out_max, _ = LE_grid(anode, x_amount=le_density, time_interval=time_interval)
        le_grids[reg_pert].append(out_max)
print('Done.')

In [ ]:
importlib.reload(plots.plots)
from plots.plots import plot_FTLEs

vmin = float(output_max_control.min())
vmax = float(output_max_control.max())
print(f'Colour scale: vmin={vmin:.3f}, vmax={vmax:.3f}')

plot_FTLEs(output_max_control, title='Control', filename=subfolder + '/FTLE_control', vmin=vmin, vmax=vmax)
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    for out_max, lbl, safe in zip(le_grids[reg_pert], interval_labels, interval_safe):
        plot_FTLEs(out_max, title=f'Flossing {lbl} ({lbl_p})',
                   filename=subfolder + f'/FTLE_flossed_{safe}_{safe_p}', vmin=vmin, vmax=vmax)

control_html = (
    f'<figure style="margin:4px"><figcaption style="font-size:10px">control</figcaption>'
    f'<img src="{subfolder}/FTLE_control.png?{time.time()}" width="280"></figure>'
)
imgs_html = ''
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    imgs_html += ''.join(
        f'<figure style="margin:4px"><figcaption style="font-size:10px">flossing {lbl} ({lbl_p})</figcaption>'
        f'<img src="{subfolder}/FTLE_flossed_{safe}_{safe_p}.png?{time.time()}" width="280"></figure>'
        for lbl, safe in zip(interval_labels, interval_safe)
    )
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{control_html}{imgs_html}</div>'))

## Attack visualisation — before vs after

In [ ]:
importlib.reload(plots.plots)
from plots.plots import classification_levelsets_with_attacks, find_attacks

# Victim attack on the control model
classification_levelsets_with_attacks(
    anode_control,
    X_base[[best_idx]], y_base[[best_idx]],
    grad=grad_best[[best_idx]],
    fig_name=subfolder + '/attack_before',
    footnote='victim attack on control model'
)

attack_counts = {reg_pert: {} for reg_pert in reg_perturbations}
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    for anode, lbl, safe in zip(anodes_flossed[reg_pert], interval_labels, interval_safe):
        X_b_f, y_b_f, g_f, stats_f = find_attacks(
            anode, X_test, y_test,
            adversarial_budget=adversarial_budget,
            attack_type='equi', equi_amount=equi_amount,
            margin=margin, return_stats=True, verbose=False
        )
        attack_counts[reg_pert][lbl] = stats_f['n_high_conf']
        if len(X_b_f) > 0:
            classification_levelsets_with_attacks(
                anode, X_b_f, y_b_f, grad=g_f,
                fig_name=subfolder + f'/attack_after_{safe}_{safe_p}',
                footnote=f'attacks after flossing {lbl} ({lbl_p})'
            )
        else:
            print(f'No attacks remaining for flossing {lbl} ({lbl_p})')

print(f'High-conf attacks — control: {stats["n_high_conf"]}')
for reg_pert in reg_perturbations:
    lbl_p = reg_pert_labels[reg_pert]
    for lbl, n in attack_counts[reg_pert].items():
        print(f'                   flossing {lbl} ({lbl_p}): {n}')

before_html = (
    f'<figure style="margin:4px"><figcaption style="font-size:10px">control</figcaption>'
    f'<img src="{subfolder}/attack_before.png?{time.time()}" width="280"></figure>'
)
imgs_html = ''
for reg_pert in reg_perturbations:
    safe_p = reg_pert_safe[reg_pert]
    lbl_p  = reg_pert_labels[reg_pert]
    imgs_html += ''.join(
        f'<figure style="margin:4px"><figcaption style="font-size:10px">after flossing {lbl} ({lbl_p})</figcaption>'
        f'<img src="{subfolder}/attack_after_{safe}_{safe_p}.png?{time.time()}" width="280"></figure>'
        for lbl, safe in zip(interval_labels, interval_safe)
    )
display(HTML(f'<div style="display:flex;flex-wrap:wrap">{before_html}{imgs_html}</div>'))

## FTLE at victim point — before vs after

In [ ]:
from FTLE import LEs

time_interval = torch.tensor([0., 10.])
les_control = LEs(victim_point, anode_control, time_interval=time_interval)
ftle_control = les_control[0].item()

ftle_after = {reg_pert: [] for reg_pert in reg_perturbations}
for reg_pert in reg_perturbations:
    for anode in anodes_flossed[reg_pert]:
        les = LEs(victim_point, anode, time_interval=time_interval)
        ftle_after[reg_pert].append(les[0].item())

labels_bar = ['control']
values_bar = [ftle_control]
colors_bar = ['C0']
color_cycle = ['C1', 'C2', 'C3', 'C4']
for ci, reg_pert in enumerate(reg_perturbations):
    short = reg_pert_safe[reg_pert]
    for lbl, v in zip(interval_labels, ftle_after[reg_pert]):
        labels_bar.append(f'flosss\n{lbl}\n({short})')
        values_bar.append(v)
        colors_bar.append(color_cycle[ci % len(color_cycle)])

print(f'Max FTLE at victim — control: {ftle_control:.4f}')
for reg_pert in reg_perturbations:
    lbl_p = reg_pert_labels[reg_pert]
    for lbl, v in zip(interval_labels, ftle_after[reg_pert]):
        print(f'                   flossing {lbl} ({lbl_p}): {v:.4f}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(labels_bar, values_bar, color=colors_bar, edgecolor='black', linewidth=0.7)
ax.axhline(le_threshold, color='red', linestyle='--', linewidth=0.9, label=f'threshold={le_threshold}')
ax.set_ylabel('Max FTLE at victim point')
ax.set_title('FTLE reduction via targeted flossing')
ax.legend(fontsize=7)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(subfolder + '/ftle_comparison.png', dpi=200, bbox_inches='tight', facecolor='white')
plt.show()